# Serve google/medgemma-1.5-4b-it on a free Colab T4

**Before running:** `Runtime → Change runtime type → T4 GPU`.

This notebook loads MedGemma directly via `transformers`
(`AutoProcessor` + `AutoModelForMultimodalLM`) —
`transformers` load (the pattern the model card itself documents) works
reliably instead, so that's what this notebook wraps in a small FastAPI
server.

**Important:** T4 is a Turing-generation GPU and does **not** support
`flash_attention_2` (that needs Ampere or newer) — the official model card
example uses it, but this notebook deliberately omits it, matching what's
actually been verified to run on a T4.

This notebook:
1. Installs dependencies
2. Logs into HuggingFace (optional but recommended, for higher rate limits)
3. Loads Medgemma-1.5-4b-it in bf16 with `device_map="auto"`
4. Wraps it in a small FastAPI server exposing `POST /analyze`
5. Runs that server in a background thread and tunnels it out with ngrok
6. Prints the public URL to paste into `MEDGEMMA_BASE_URL` in your project's `.env`

**Note:** Colab free-tier sessions are ephemeral — this is for demos/dev,
not something to depend on for production uptime.

## 1. Confirm GPU is attached

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio

## 3. Log into HuggingFace (recommended)
Paste a token with **read** access (create one at https://huggingface.co/settings/tokens).
Medgemma-1.5-4b-it requires accepting a license.

In [ ]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Paste your HuggingFace token (or leave blank to skip): ")
if hf_token.strip():
    login(token=hf_token)
else:
    print("Skipping HF login — unauthenticated requests, lower rate limits.")

## 4. Set your ngrok authtoken
Free account: sign up at https://dashboard.ngrok.com/signup, then copy your authtoken from
https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok, conf

ngrok_token = getpass("Paste your ngrok authtoken: ")
conf.get_default().auth_token = ngrok_token

## 5. Load medgemma-1.5-4b-it
This is the exact loading pattern verified to work on a T4: bf16, `device_map="auto"`,and **no** `attn_implementation` override (letting transformers
pick a T4-compatible default rather than the official example's `flash_attention_2`,
which T4 cannot run). First run downloads the weights (~8GB), so this can take a few minutes.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/medgemma-1.5-4b-it"

processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it")
print("Processor loaded successfully.")
model = AutoModelForMultimodalLM.from_pretrained("google/medgemma-1.5-4b-it",dtype=torch.bfloat16 , device_map="auto")
model.eval()

print("medgemma loaded successfully.")

## 6. Define the FastAPI server
Exposes `POST /analyze` accepting `{"image_b64", "mime_type", "prompt", "max_new_tokens", "temperature", "top_p"}`
and returning `{"text", "confidence"}` — this is the exact contract `medgemma_client.py` in the
backend expects. The generation call itself mirrors the tested pattern; the only addition is
`return_dict_in_generate=True, output_scores=True` so a confidence estimate (mean max-softmax
probability across generated tokens) can be computed without changing the sampling behavior.

In [ ]:
import base64
import traceback
from io import BytesIO

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from PIL import Image

app = FastAPI(title="medgemma-1.5-4b-itinference server")


class AnalyzeRequest(BaseModel):
    image_b64: str
    mime_type: str = "image/png"
    prompt: str = "Analyze this medical image and describe any notable findings."
    max_new_tokens: int = 512
    temperature: float = 0.2
    top_p: float = 0.8


class AnalyzeResponse(BaseModel):
    text: str
    confidence: float


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}


@app.post("/analyze", response_model=AnalyzeResponse)
def analyze(req: AnalyzeRequest):
    try:
        image_bytes = base64.b64decode(req.image_b64)
        image = Image.open(BytesIO(image_bytes)).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": req.prompt},
                ],
            }
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        input_length = inputs["input_ids"].shape[-1]

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=req.max_new_tokens,
                temperature=req.temperature,
                do_sample=True,
                top_p=req.top_p,
                return_dict_in_generate=True,
                output_scores=True,
            )

        generated_tokens = outputs.sequences[0][input_length:]
        text = processor.decode(generated_tokens, skip_special_tokens=True)

        # Confidence: mean max-softmax-probability across generated tokens.
        # Purely additive — doesn't change the sampling/generation behavior above.
        try:
            step_confidences = [
                torch.softmax(step_logits[0], dim=-1).max().item()
                for step_logits in outputs.scores
            ]
            confidence = sum(step_confidences) / len(step_confidences) if step_confidences else 0.5
        except Exception:
            confidence = 0.5

        return AnalyzeResponse(text=text, confidence=confidence)
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

## 7. Run the server in a background thread
Runs on port 8001, inside this notebook's process — reuses the already-loaded
`model`/`processor` directly rather than reloading them in a separate process.

In [ ]:
import threading
import time
import requests
import nest_asyncio
import uvicorn

nest_asyncio.apply()

def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8001, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

print("Waiting for the server to start listening...")
for attempt in range(30):
    try:
        r = requests.get("http://localhost:8001/health", timeout=2)
        if r.status_code == 200:
            print("Server is up:", r.json())
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    print("Server did not start within 30s — check for errors above.")

## 8. Open the public tunnel

In [ ]:
public_tunnel = ngrok.connect(8001, "http")
public_url = public_tunnel.public_url

print("=" * 70)
print("MEDGEMMA is live at:", public_url)
print()
print("Paste this into your backend .env file:")
print(f"MEDGEMMA_BASE_URL={public_url}")
print("=" * 70)

## 9. Sanity-check the endpoint
Sends a real request with a public chest X-ray image, exactly like `medgemma_client.py` does.

In [ ]:
import base64
import requests

# Upload/select your image
from google.colab import files
uploaded = files.upload()

image_path = next(iter(uploaded))

# Read your uploaded image
with open(image_path, "rb") as f:
    image_bytes = f.read()

In [ ]:
import base64
import requests

image_b64 = base64.b64encode(image_bytes).decode("utf-8")

# Detect MIME type from the filename
import mimetypes
mime_type = mimetypes.guess_type(image_path)[0] or "image/png"

response = requests.post(
    f"{public_url}/analyze",
    json={
        "image_b64": image_b64,
        "mime_type": "image/png",
        "prompt": "Describe any notable findings in this image.",
    },
    timeout=120,
)
response.raise_for_status()
result = response.json()
print("Findings:", result["text"])
print("Confidence:", result["confidence"])

## Notes
- Keep this notebook open/running for the duration of your demo — closing it or letting Colab time out kills the server and tunnel.
- The free ngrok URL changes every time you rerun cell 8 (restart the tunnel), so re-copy it into `.env` and restart your backend if you restart this notebook.
- Colab free tier disconnects after ~90 min idle or ~12 hrs total — start this ~15-20 min before you need it live, not cold.
- If `/analyze` errors, check the notebook's own cell output — the server logs the full traceback there via `traceback.print_exc()`.